In [1]:
'''
Train a Transformer encoder model to generate MIDI:
    1. pre-train
    (2. SFT)
    (3. RL)
'''
from pathlib import Path
from random import shuffle

from miditok import REMI, TokenizerConfig
from miditok.utils import split_files_for_training

from config import ModelParams, HyperParams

PREPROCESS = False # set to True if you haven't run PREPROCESS yet
TRAIN = True # set to False if you want to skip training

TOKENIZER_PARAMS = {
    "special_tokens": ["PAD_None", "MASK_None"],
    "pitch_range": (22, 107),
    "num_velocities": 32,
    "beat_res": {(0, 8): 4}, # 16 Position tokens within a bar
    "use_pitchdrum_tokens": False,
    "use_chords": False,
    "chord_tokens_with_root_note": False,
}
config = TokenizerConfig(**TOKENIZER_PARAMS)
tokenizer = REMI(config)

if PREPROCESS:
    midi_paths = list(Path(HyperParams.DATA_PATH).rglob("*.mid"))
    shuffle(midi_paths)
    midi_paths = [p.resolve() for p in midi_paths if p.is_file()]
    total_num_files = len(midi_paths)

    tokenizer.save(Path("models", HyperParams.name + "_tokenizer.json"))

    num_files_valid = round(total_num_files * 0.05)
    num_files_test = round(total_num_files * 0.05)
    midi_paths_val = midi_paths[:num_files_valid]
    midi_paths_test = midi_paths[num_files_valid:num_files_valid + num_files_test]
    midi_paths_train = midi_paths[num_files_valid + num_files_test:]
    
    # Chunk MIDIs and perform data augmentation on each subset independently
    for files_paths, subset_name in (
        (midi_paths_train, "train"), (midi_paths_val, "val"), (midi_paths_test, "test")
    ):
        subset_chunks_dir = Path(f"pre-train_dataset/{subset_name}")
        split_files_for_training(
            files_paths=files_paths,
            tokenizer=tokenizer,
            save_dir=subset_chunks_dir,
            max_seq_len=1024,
            num_overlap_bars=2,
        )
else:
    tokenizer._load_from_json(Path("models", HyperParams.name + "_tokenizer.json"))

print(tokenizer.vocab)
print(tokenizer.vocab_size)

{'PAD_None': 0, 'MASK_None': 1, 'Bar_None': 2, 'Pitch_22': 3, 'Pitch_23': 4, 'Pitch_24': 5, 'Pitch_25': 6, 'Pitch_26': 7, 'Pitch_27': 8, 'Pitch_28': 9, 'Pitch_29': 10, 'Pitch_30': 11, 'Pitch_31': 12, 'Pitch_32': 13, 'Pitch_33': 14, 'Pitch_34': 15, 'Pitch_35': 16, 'Pitch_36': 17, 'Pitch_37': 18, 'Pitch_38': 19, 'Pitch_39': 20, 'Pitch_40': 21, 'Pitch_41': 22, 'Pitch_42': 23, 'Pitch_43': 24, 'Pitch_44': 25, 'Pitch_45': 26, 'Pitch_46': 27, 'Pitch_47': 28, 'Pitch_48': 29, 'Pitch_49': 30, 'Pitch_50': 31, 'Pitch_51': 32, 'Pitch_52': 33, 'Pitch_53': 34, 'Pitch_54': 35, 'Pitch_55': 36, 'Pitch_56': 37, 'Pitch_57': 38, 'Pitch_58': 39, 'Pitch_59': 40, 'Pitch_60': 41, 'Pitch_61': 42, 'Pitch_62': 43, 'Pitch_63': 44, 'Pitch_64': 45, 'Pitch_65': 46, 'Pitch_66': 47, 'Pitch_67': 48, 'Pitch_68': 49, 'Pitch_69': 50, 'Pitch_70': 51, 'Pitch_71': 52, 'Pitch_72': 53, 'Pitch_73': 54, 'Pitch_74': 55, 'Pitch_75': 56, 'Pitch_76': 57, 'Pitch_77': 58, 'Pitch_78': 59, 'Pitch_79': 60, 'Pitch_80': 61, 'Pitch_81': 62, 

In [2]:
from help import MIDIDataset, DataCollator
from torch.utils.data import DataLoader

midi_paths_train = list(Path("pre-train_dataset", "train").rglob("*.mid"))
midi_paths_val = list(Path("pre-train_dataset", "val").rglob("*.mid"))

train_dataset = MIDIDataset(midi_paths_train, tokenizer)
val_dataset = MIDIDataset(midi_paths_val, tokenizer)

# Extend sequences with PAD tokens (PAD tokens are treated as normal tokens during training)
collator = DataCollator(pad_token_id=tokenizer.vocab['PAD_None'])

train_dataloader = DataLoader(dataset=train_dataset, batch_size=HyperParams.batch_size, collate_fn=collator, shuffle=True)
val_dataloader = DataLoader(dataset=val_dataset, batch_size=HyperParams.batch_size, collate_fn=collator, shuffle=True)

print(len(train_dataset), "train samples")
print(len(val_dataset), "validation samples")

8362 train samples
457 validation samples


In [3]:
from help import Trainer
from tqdm.notebook import tqdm

trainer = Trainer(ModelParams, HyperParams, tokenizer)

print(sum(p.numel() for p in trainer.mask_predictor.parameters()), "model parameters\n")
print("Model architecture:")
print(trainer.mask_predictor)

if HyperParams.retrain:
    # load model and its optimizer from a checkpoint (fine_tune=True -> load only model)
    loss = trainer.load("models/checkpoints/pre-training_792000_checkpoint.pth", fine_tune=True)

c:\Users\ville\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\nn\modules\transformer.py:379: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(


31639720 model parameters

Model architecture:
MaskPredictor(
  (token_emb): TokenEmbedding(
    (emb_lookup): Embedding(168, 512)
  )
  (pos_emb): PositionalEncoding()
  (emb_dropout): Dropout(p=0.1, inplace=False)
  (transformer): TransformerEncoder(
    (layers): ModuleList(
      (0-9): 10 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=512, out_features=512, bias=False)
        )
        (linear1): Linear(in_features=512, out_features=2048, bias=False)
        (dropout): Dropout(p=0.1, inplace=False)
        (linear2): Linear(in_features=2048, out_features=512, bias=False)
        (norm1): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
        (norm2): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
        (dropout1): Dropout(p=0.1, inplace=False)
        (dropout2): Dropout(p=0.1, inplace=False)
      )
    )
  )
  (lm_head): Linear(in_features=512, out_features=168, bias=True)
)


In [ ]:
if TRAIN:
    from torch.utils.tensorboard import SummaryWriter
    from itertools import cycle
    
    # Setup
    trainer.phase = 'pre-training' # 'pre-training' or 'SFT'
    model_dir = 'models/checkpoints/' + trainer.phase + '_'

    writer = SummaryWriter()

    global_step = 0

    # Optional: wrap dataloader with cycle to avoid worrying about epoch ends
    train_iterator = cycle(train_dataloader)

    pbar = tqdm(total=HyperParams.num_training_steps, desc="Training", position=0)

    best_val = float('inf')
    train_loss = 0

    trainer.mask_predictor.train()
    while global_step < HyperParams.num_training_steps:
        train_batch = next(train_iterator)

        train_loss += trainer.train_step(train_batch)

        # Logging
        global_step += 1
        if global_step % HyperParams.logging_interval == 0:
            val_loss = 0

            trainer.mask_predictor.eval()
            for val_batch in tqdm(val_dataloader, desc="Validating", leave=False):
                val_loss += trainer.val_step(val_batch)
            trainer.mask_predictor.train()

            writer.add_scalar('Loss/val', val_loss / len(val_dataloader), global_step)       
            writer.add_scalar('Loss/train', train_loss / HyperParams.logging_interval, global_step)
            writer.add_scalar('LR', trainer.lr_scheduler.get_last_lr()[0], global_step)

            if val_loss < best_val:
                trainer.save('models/best_val', val_loss / len(val_dataloader))
                best_val = val_loss

            trainer.save(model_dir + str(global_step), train_loss / HyperParams.logging_interval)
            train_loss = 0

            trainer.mask_predictor.train()
        pbar.update(1)

    pbar.close()
    writer.close()

Training:   0%|          | 0/40000000 [00:00<?, ?it/s]

c:\Users\ville\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\optim\lr_scheduler.py:224: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  warnings.warn(


Validating:   0%|          | 0/58 [00:00<?, ?it/s]